# 01 · Auditoría de datos para el ranker de transacciones

**Pregunta de fondo:** ¿el material base (`customers`, `transactions`) es lo bastante realista para
construir consultas de reclamo sobre transacciones reales y que ordenar candidatas no sea trivial?

Todo se calcula con DuckDB sobre disco (base limpia, solo lectura). Semilla 42 en todo muestreo.
Ningún dato sale de la máquina.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import numpy as np, pandas as pd
from common import connect, DB_PATH, SEED, DISPUTABLE_SQL, DISPUTABLE_TYPES
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 80)
con = connect()
q = lambda s: con.sql(s).df()
print(DB_PATH.name, "| semilla", SEED)

bank.duckdb | semilla 42


## 1. Esquema real, nulos, duplicados y rango de fechas
**Pregunta:** ¿qué columnas existen realmente y en qué estado están? No se asume ninguna columna
que no aparezca aquí.

In [2]:
for t in ["customers", "transactions"]:
    d = q(f"DESCRIBE {t}")[["column_name", "column_type"]]
    n = con.sql(f"SELECT count(*) FROM {t}").fetchone()[0]
    nulls = con.sql("SELECT " + ", ".join(f"count(*) - count(\"{c}\") AS \"{c}\"" for c in d.column_name) + f" FROM {t}").df().T[0]
    d["pct_null"] = (d.column_name.map(nulls) / n * 100).round(2).values
    print(f"\n=== {t}: {n:,} filas, {len(d)} columnas")
    print(d.to_string(index=False))


=== customers: 150,000 filas, 28 columnas
             column_name column_type  pct_null
             customer_id     VARCHAR      0.00
         document_number     VARCHAR      0.00
           document_type     VARCHAR      0.00
              first_name     VARCHAR      0.00
               last_name     VARCHAR      0.00
           date_of_birth        DATE      0.00
                  gender     VARCHAR      0.00
                   email     VARCHAR      1.99
            mobile_phone     VARCHAR      3.14
          landline_phone     VARCHAR     50.04
                 address     VARCHAR      4.91
                    city     VARCHAR      0.00
                   state     VARCHAR      0.00
                 country     VARCHAR      0.00
             postal_code     VARCHAR     10.03
         detected_accent     VARCHAR     29.88
                 segment     VARCHAR      0.00
            credit_score      BIGINT     14.99
estimated_monthly_income      DOUBLE     20.02
              occ

In [3]:
print(q("""
SELECT 'transactions' AS tabla, count(*) AS filas, count(DISTINCT transaction_id) AS pk_distintas,
       (SELECT count(*) FROM raw_transactions) AS filas_raw,
       min(transaction_date) AS desde, max(transaction_date) AS hasta,
       count(DISTINCT customer_id) AS clientes_con_tx
FROM transactions
UNION ALL
SELECT 'customers', count(*), count(DISTINCT customer_id), (SELECT count(*) FROM raw_customers),
       min(registration_date), max(registration_date), NULL FROM customers
""").to_string(index=False))
# duplicados por clave de negocio: mismo cliente, instante, monto y tipo
print(q("""
SELECT count(*) AS grupos_duplicados, coalesce(sum(n - 1), 0) AS filas_sobrantes FROM (
  SELECT customer_id, transaction_date, amount, transaction_type, count(*) n
  FROM transactions GROUP BY ALL HAVING count(*) > 1)
""").to_string(index=False))
print(q("SELECT count(*) AS tx_sin_cliente_en_customers FROM transactions t ANTI JOIN customers c USING (customer_id)").to_string(index=False))

       tabla   filas  pk_distintas  filas_raw               desde               hasta  clientes_con_tx
transactions 4425008       4425008    4425008 2023-06-17 06:01:30 2026-06-18 05:59:41           134515
   customers  150000        150000     150000 2018-06-18 01:21:11 2026-06-17 23:53:29             <NA>
 grupos_duplicados  filas_sobrantes
                 0              0.0
 tx_sin_cliente_en_customers
                           0


**Interpretación:** La base limpia conserva las 4.425.008 transacciones del crudo (0 PK duplicadas, 0 duplicados por cliente+instante+monto+tipo, 0 transacciones huérfanas) y cubre 2023-06-17 → 2026-06-18 (3 años) para 134.515 clientes. Los nulos son estructurales: `merchant_name` falta en el 76,7 % porque solo existe en compras, y `amount_usd` en el 57 % porque las transacciones en USD no la traen. El material es consistente, pero eso no dice nada aún sobre si es realista.

## 2. Comercios
**Pregunta:** ¿`merchant_name` parece un descriptor bancario real ("UBER *TRIP", "MP*…") o un catálogo
corto de nombres limpios? ¿La relación comercio ↔ `merchant_category` ↔ `transaction_category` es
consistente o aleatoria?

In [4]:
print(q("""
SELECT transaction_type, count(*) AS n, round(avg((merchant_name IS NOT NULL)::INT)*100, 1) AS pct_con_comercio,
       count(DISTINCT merchant_name) AS comercios_distintos,
       round(avg((transaction_category IS NOT NULL)::INT)*100, 1) AS pct_con_tx_category
FROM transactions GROUP BY 1 ORDER BY n DESC""").to_string(index=False))
top = q("""
SELECT merchant_name, count(*) AS n, length(merchant_name) AS largo,
       any_value(merchant_category) FILTER (WHERE merchant_category IS NOT NULL) AS merchant_category
FROM transactions WHERE merchant_name IS NOT NULL GROUP BY 1 ORDER BY n DESC LIMIT 30""")
print(f"\nComercios distintos: {len(top)} (se pidió top 30)")
print(top.to_string(index=False))
print("\nlargo medio:", top.largo.mean().round(1),
      "| con '*' o dígitos:", top.merchant_name.str.contains(r"[*0-9]").sum(),
      "| en mayúsculas:", (top.merchant_name == top.merchant_name.str.upper()).sum())

transaction_type       n  pct_con_comercio  comercios_distintos  pct_con_tx_category
        Purchase 1083406              95.0                   24                 95.0
      Withdrawal  964673               0.0                    0                  0.0
        Transfer  896438               0.0                    0                  0.0
         Payment  738964               0.0                    0                 95.0
         Deposit  609409               0.0                    0                  0.0
      Adjustment  132118               0.0                    0                  0.0

Comercios distintos: 24 (se pidió top 30)
            merchant_name     n  largo merchant_category
             Super Ahorro 64527     12              Food
Restaurante El Buen Sabor 64370     25              Food
          Tienda Don José 64249     15              Food
          Mercado Central 63912     15              Food
       Empresa Telefónica 51464     18          Services
                 Cab

In [5]:
# consistencia: cada comercio, ¿cuántas categorías tiene? y merchant_category vs transaction_category
print(q("""
SELECT merchant_name, count(DISTINCT merchant_category) AS n_merchant_cat, count(DISTINCT transaction_category) AS n_tx_cat,
       count(DISTINCT channel) AS n_canales, count(DISTINCT transaction_country) AS n_paises
FROM transactions WHERE merchant_name IS NOT NULL GROUP BY 1 ORDER BY 1""").describe().loc[["min", "max"]].to_string())
print(q("""
SELECT merchant_category, transaction_category, count(*) n
FROM transactions WHERE transaction_type='Purchase' AND merchant_category IS NOT NULL AND transaction_category IS NOT NULL
GROUP BY ALL ORDER BY 1, 2""").pivot(index="merchant_category", columns="transaction_category", values="n").fillna(0).astype(int).to_string())

     n_merchant_cat  n_tx_cat  n_canales  n_paises
min             1.0       1.0        6.0       6.0
max             1.0       1.0        6.0       6.0
transaction_category  Entertainment    Food  Health   Other  Services  Transport
merchant_category                                                               
Entertainment                146307       0       0       0         0          0
Food                              0  243898       0       0         0          0
Health                            0       0   97764       0         0          0
Other                             0       0       0  147373         0          0
Services                          0       0       0       0    194842          0
Transport                         0       0       0       0         0     147275


In [6]:
# ¿el comercio depende del cliente (hábitos) o se sortea en cada transacción?
# Comparamos comercios distintos por cliente contra lo esperado si se sortearan al azar con la
# frecuencia global (simulación con semilla fija).
per_cust = q("""
SELECT customer_id, count(*) AS n, count(DISTINCT merchant_name) AS distintos
FROM transactions WHERE merchant_name IS NOT NULL GROUP BY 1""")
freq = top.set_index("merchant_name").n; p = (freq / freq.sum()).values
rng = np.random.default_rng(SEED)
samp = per_cust.sample(5000, random_state=SEED)
sim = [len(np.unique(rng.choice(len(p), size=k, p=p))) for k in samp.n]
print(pd.DataFrame({"observado": samp.distintos.describe(), "si_fuera_al_azar": pd.Series(sim).describe()}).round(2).T.to_string())

                   count  mean   std  min  25%  50%   75%   max
observado         5000.0  9.21  3.72  1.0  6.0  8.0  12.0  23.0
si_fuera_al_azar  5000.0  9.24  3.74  1.0  6.0  8.0  12.0  23.0


**Interpretación:** Es un **catálogo corto de nombres limpios**: 24 comercios, solo en `Purchase` (95 %), 4 por categoría, sin asteriscos, dígitos ni mayúsculas de descriptor bancario. Cada comercio tiene exactamente 1 `merchant_category`, y `merchant_category` = `transaction_category` en el 100 % de los casos con ambos campos: la relación es determinista, no aleatoria. Pero el comercio no depende del cliente (comercios distintos por cliente: 9,29 observado vs 9,29 sorteando al azar) ni del canal/país (cada comercio aparece en los 6 canales y 6 países). Consecuencia: la similitud de comercio es trivial (24 nombres casi sin confusiones posibles) y solo discrimina dentro de las compras. No hay hábitos de cliente que aprender.

## 3. Montos
**Pregunta:** ¿cómo se distribuyen por moneda y tipo? ¿hay montos redondos repetidos (p. ej. 120.00)
como en la vida real? ¿`amount_usd` es coherente con `amount` y `daily_exchange_rates`?

In [7]:
amt = q(f"""
SELECT currency, transaction_type, count(*) AS n,
       round(min(amount), 2) AS min, round(quantile_cont(amount, 0.25), 2) AS p25, round(median(amount), 2) AS p50,
       round(quantile_cont(amount, 0.75), 2) AS p75, round(max(amount), 2) AS max
FROM transactions WHERE {DISPUTABLE_SQL} GROUP BY ALL ORDER BY 1, 2""")
print(amt.to_string(index=False))
# uniformidad: en una U(a,b) p25, p50, p75 están a 25/50/75 % del rango
amt["pos_p25"] = ((amt.p25 - amt["min"]) / (amt["max"] - amt["min"])).round(3)
amt["pos_p50"] = ((amt.p50 - amt["min"]) / (amt["max"] - amt["min"])).round(3)
amt["pos_p75"] = ((amt.p75 - amt["min"]) / (amt["max"] - amt["min"])).round(3)
print("\nposición relativa de los cuartiles dentro de [min, max] (uniforme ⇒ 0.25/0.50/0.75):")
print(amt[["currency", "transaction_type", "pos_p25", "pos_p50", "pos_p75"]].to_string(index=False))

currency transaction_type      n       min        p25        p50        p75        max
     ARS          Payment 132416  17500.08  187875.12  358923.76  529740.17  699998.08
     ARS         Purchase 194216   1750.05   45153.56   88306.28  131820.47  174999.31
     ARS       Withdrawal 172622   7002.26   48870.73   90848.78  132976.70  174998.86
     COP          Payment 198867 200125.88 2149290.53 4104373.65 6048381.05 7999997.10
     COP         Purchase 292781  20006.90  514382.77 1010984.32 1508370.91 1999997.03
     COP       Withdrawal 260708  80002.46  557983.56 1038505.52 1517419.89 1999998.29
     USD          Payment 407681     50.00     537.41    1025.79    1511.97    2000.00
     USD         Purchase 596409      5.00     128.25     252.14     376.10     500.00
     USD       Withdrawal 531343     20.00     140.13     260.21     380.33     500.00

posición relativa de los cuartiles dentro de [min, max] (uniforme ⇒ 0.25/0.50/0.75):
currency transaction_type  pos_p25  pos_p50 

In [8]:
# montos redondos y repetidos
print(q(f"""
SELECT currency,
  round(avg((amount = round(amount))::INT)*100, 2) AS pct_sin_centavos,
  round(avg((amount % 10 = 0)::INT)*100, 3) AS pct_multiplo_10,
  round(avg((amount % 100 = 0)::INT)*100, 4) AS pct_multiplo_100,
  round(avg((round(amount*100) % 100 = 99)::INT)*100, 2) AS pct_termina_99,
  count(*) AS n
FROM transactions WHERE {DISPUTABLE_SQL} GROUP BY 1""").to_string(index=False))
print("(en un sorteo uniforme de centavos: sin centavos = 1 %, termina en .99 = 1 %)")
print(q(f"""
SELECT count(*) AS pares_mismo_cliente_mismo_monto FROM (
  SELECT customer_id, amount, count(*) n FROM transactions WHERE {DISPUTABLE_SQL} GROUP BY ALL HAVING n > 1)""").to_string(index=False))
print(q("SELECT amount, count(*) n FROM transactions WHERE currency='USD' AND amount IN (120, 100, 50, 19.99, 9.99) GROUP BY 1 ORDER BY 1").to_string(index=False))

currency  pct_sin_centavos  pct_multiplo_10  pct_multiplo_100  pct_termina_99       n
     ARS              1.01            0.100            0.0086            0.99  499254
     COP              1.00            0.096            0.0090            1.02  752356
     USD              0.99            0.102            0.0103            1.00 1535433
(en un sorteo uniforme de centavos: sin centavos = 1 %, termina en .99 = 1 %)
 pares_mismo_cliente_mismo_monto
                             279
 amount  n
   9.99 11
  19.99 18
  50.00 21
 100.00 35
 120.00 35


In [9]:
# coherencia de amount_usd (columna original) con la tasa diaria de daily_exchange_rates
fx = q("""
WITH t AS (SELECT * FROM transactions WHERE currency <> 'USD' AND amount_usd IS NOT NULL
           USING SAMPLE reservoir(200000 ROWS) REPEATABLE (42)),
r AS (SELECT date, source_currency, avg(exchange_rate) AS rate, min(exchange_rate) AS rmin, max(exchange_rate) AS rmax
      FROM daily_exchange_rates WHERE target_currency = 'USD' GROUP BY ALL)
SELECT t.currency, t.amount, t.amount_usd, r.rate, r.rmin, r.rmax,
       t.amount_usd / t.amount AS implied_rate
FROM t LEFT JOIN r ON r.source_currency = t.currency AND r.date = CAST(t.transaction_date AS DATE)""")
fx["relerr_vs_rate_dia"] = (fx.implied_rate / fx.rate - 1).abs()
fx["dentro_rango_fuentes"] = fx.implied_rate.between(fx.rmin * 0.999, fx.rmax * 1.001)
print(fx.groupby("currency").agg(n=("amount", "size"), sin_tasa=("rate", lambda s: s.isna().sum()),
      relerr_mediana=("relerr_vs_rate_dia", "median"), relerr_p95=("relerr_vs_rate_dia", lambda s: s.quantile(.95)),
      pct_dentro_rango=("dentro_rango_fuentes", "mean")).round(4).to_string())
print(q("SELECT currency, count(*) n, count(amount_usd) AS con_amount_usd FROM transactions GROUP BY 1").to_string(index=False))

              n  sin_tasa  relerr_mediana  relerr_p95  pct_dentro_rango
currency                                                               
ARS       33964         9          0.0096      0.0190            0.0587
COP       51448        15          0.0119      0.0204            0.0903
currency       n  con_amount_usd
     ARS  792585          752544
     COP 1194444         1135008
     USD 2437979               0


**Interpretación:** Los montos son **uniformes** por tipo y moneda: los cuartiles caen exactamente en 0,25/0,50/0,75 del rango (p. ej. compras USD ~U(5, 500)), y los centavos también: sin centavos, 1,0 %, y terminados en .99, 1,0 %, igual que al azar. No hay montos redondos ni precios psicológicos (120,00 USD aparece 35 veces en 1,5 M) y solo hay 279 pares cliente-monto repetidos en 2,79 M. El monto exacto identifica casi siempre una única transacción. `amount_usd` no coincide con la tasa diaria de `daily_exchange_rates`: el error relativo mediano es ~1 % y solo el 6–9 % cae dentro del rango de tasas publicado ese día, así que tiene ruido propio. Para convertir moneda se usa `amount_usd_filled` del ETL y se asume ese error de ~1–2 %.

## 4. Estados y universo disputable
**Pregunta:** ¿qué proporción hay de cada estado y qué transacciones son "cargos" que un cliente
podría disputar?

In [10]:
st = q("""
SELECT transaction_type, transaction_status, count(*) n FROM transactions GROUP BY ALL""")
piv = st.pivot(index="transaction_type", columns="transaction_status", values="n")
print((piv.div(piv.sum(axis=1), axis=0) * 100).round(2).assign(total=piv.sum(axis=1)).to_string())
print("\nglobal (%):", (st.groupby("transaction_status").n.sum() / st.n.sum() * 100).round(2).to_dict())
print(q("""SELECT transaction_type, count(DISTINCT channel) AS canales, list(DISTINCT channel ORDER BY channel) AS lista
           FROM transactions GROUP BY 1""").to_string(index=False))

transaction_status  Approved  Declined  Pending  Reversed    total
transaction_type                                                  
Adjustment             91.88      5.06     2.05      1.01   132118
Deposit                92.01      4.96     2.01      1.01   609409
Payment                92.01      4.99     1.99      1.01   738964
Purchase               91.95      5.04     1.99      1.02  1083406
Transfer               91.99      5.02     1.99      1.00   896438
Withdrawal             92.03      4.97     2.00      1.01   964673

global (%): {'Approved': 91.99, 'Declined': 5.0, 'Pending': 2.0, 'Reversed': 1.01}
transaction_type  canales                                  lista
         Payment        6 [ATM, App, Branch, POS, Transfer, Web]
        Transfer        6 [ATM, App, Branch, POS, Transfer, Web]
      Withdrawal        6 [ATM, App, Branch, POS, Transfer, Web]
         Deposit        6 [ATM, App, Branch, POS, Transfer, Web]
      Adjustment        6 [ATM, App, Branch, POS, Trans

In [11]:
univ = q(f"""
SELECT count(*) AS disputables, round(count(*) / (SELECT count(*) FROM transactions) * 100, 1) AS pct_del_total,
       sum((transaction_status IN ('Approved','Pending'))::INT) AS objetivo_posible
FROM transactions WHERE {DISPUTABLE_SQL}""")
print(univ.to_string(index=False))

 disputables  pct_del_total  objetivo_posible
     2787043           63.0         2619413.0


**Universo disputable (decisión):** `transaction_type ∈ {Purchase, Payment, Withdrawal}`. Todas tienen
monto positivo y representan dinero que sale de la cuenta a iniciativa (supuesta) del cliente:
compra en comercio, pago (con `transaction_category`) y retiro. Se excluye `Deposit` (abono),
`Transfer` (la inicia el cliente hacia un tercero; se disputa por otro flujo) y `Adjustment` (lo
aplica el banco; sin comercio ni categoría). Como **objetivo** del reclamo solo se usan estados
`Approved` y `Pending` (hay dinero afectado); `Declined` y `Reversed` se mantienen como **candidatas**
porque aparecen en el extracto y el cliente puede confundirse con ellas.

**Interpretación:** Los estados están fijados en proporciones idénticas en todos los tipos (≈92 % Approved, 5 % Declined, 2 % Pending, 1 % Reversed), y todos los tipos aparecen en los 6 canales (hay compras en ATM y retiros en POS): el canal no aporta semántica. El universo disputable queda en 2.787.043 transacciones (63 % del total), de las cuales 2.619.413 pueden ser objetivo (Approved/Pending).

## 5. Densidad de transacciones disputables por cliente
**Pregunta:** en ventanas de 7, 30 y 60 días, ¿cuántas transacciones disputables tiene el cliente?
Esto es el número de candidatas que verá el ranker.

Se ancla en una muestra de 100k transacciones disputables (semilla 42) y se cuentan las
disputables del mismo cliente en los W días anteriores (incluida la propia). Así la distribución
está ponderada como los reclamos: un cliente con más transacciones genera más reclamos.

In [12]:
con.execute(f"""CREATE OR REPLACE TEMP TABLE disp AS
  SELECT transaction_id, customer_id, transaction_date, amount FROM transactions WHERE {DISPUTABLE_SQL}""")
dens = q("""
WITH a AS (SELECT * FROM disp USING SAMPLE reservoir(100000 ROWS) REPEATABLE (42)),
     w(w) AS (VALUES (7), (30), (60))
SELECT a.transaction_id, w.w, count(*) AS n
FROM a CROSS JOIN w JOIN disp b ON b.customer_id = a.customer_id
 AND b.transaction_date BETWEEN a.transaction_date - to_days(w.w) AND a.transaction_date
GROUP BY ALL""")
summary = dens.groupby("w").n.describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(2)
summary["pct_1_sola"] = dens.groupby("w").n.apply(lambda s: (s == 1).mean() * 100).round(1)
summary["pct_ge5"] = dens.groupby("w").n.apply(lambda s: (s >= 5).mean() * 100).round(1)
print(summary.to_string())
print("\nDistribución del nº de candidatas en 60 días (%):")
print((dens[dens.w == 60].n.clip(upper=10).value_counts(normalize=True).sort_index() * 100).round(1).rename(index={10: "10+"}).to_string())

       count  mean   std  min  10%  25%  50%  75%  90%  99%   max  pct_1_sola  pct_ge5
w                                                                                     
7   100000.0  1.18  0.43  1.0  1.0  1.0  1.0  1.0  2.0  3.0   6.0        83.9      0.0
30  100000.0  1.77  0.96  1.0  1.0  1.0  2.0  2.0  3.0  5.0   8.0        49.7      1.5
60  100000.0  2.52  1.46  1.0  1.0  1.0  2.0  3.0  4.0  7.0  13.0        28.3      9.8

Distribución del nº de candidatas en 60 días (%):
n
1      28.3
2      29.8
3      20.6
4      11.6
5       5.6
6       2.6
7       1.0
8       0.4
9       0.1
10+     0.1


In [13]:
# por cliente: disputables por mes activo
pc = q("""SELECT customer_id, count(*) AS n, date_diff('day', min(transaction_date), max(transaction_date)) AS dias
          FROM disp GROUP BY 1""")
pc["por_30d"] = pc.n / (pc.dias.clip(lower=30) / 30)
print(pc[["n", "dias", "por_30d"]].describe(percentiles=[.1, .5, .9, .99]).round(2).to_string())

               n       dias    por_30d
count  134416.00  134416.00  134416.00
mean       20.73     937.84       0.64
std        13.38     174.33       0.36
min         1.00       0.00       0.06
10%         6.00     732.00       0.24
50%        18.00     997.00       0.57
90%        39.00    1071.00       1.14
99%        61.00    1090.00       1.75
max       118.00    1097.00       3.32


**Interpretación:** La densidad es **muy baja**: un cliente tiene en promedio 0,64 disputables cada 30 días. En la ventana de 60 días que usará el ranker, la mediana es de 2 candidatas, el 28,3 % de los reclamos tendría una sola candidata (trivial) y solo el 9,8 % tendría 5 o más. En 7 días, el 83,9 % tiene una sola. Con tan pocas candidatas, casi cualquier pista (incluso la recencia) desempata: el techo de dificultad del problema lo pone la densidad, no el ranker.

## 6. Coherencia temporal y geográfica
**Pregunta:** ¿el país y la ciudad de la transacción se relacionan con los del cliente? ¿hay patrones
horarios o semanales, o de monto según hora/día/comercio? Si todo es uniforme, el ranker no podrá
aprender de eso.

In [14]:
geo = q("""
WITH t AS (SELECT * FROM transactions USING SAMPLE reservoir(300000 ROWS) REPEATABLE (42))
SELECT round(avg((t.transaction_country = t.customer_country)::INT)*100, 2) AS pct_mismo_pais,
       round(avg((t.transaction_city = c.city)::INT) FILTER (WHERE t.transaction_city IS NOT NULL AND c.city IS NOT NULL)*100, 2) AS pct_misma_ciudad,
       count(DISTINCT t.transaction_city) AS ciudades_tx, count(DISTINCT c.city) AS ciudades_cliente
FROM t JOIN customers c USING (customer_id)""")
print(geo.to_string(index=False))
# desglose: país/ciudad de la transacción frente a los del cliente
print(q("""
WITH t AS (SELECT * FROM transactions USING SAMPLE reservoir(300000 ROWS) REPEATABLE (42))
SELECT CASE WHEN t.transaction_country <> t.customer_country THEN 'otro país'
            WHEN t.transaction_city IS NULL THEN 'mismo país, ciudad nula'
            WHEN t.transaction_city = c.city THEN 'mismo país, misma ciudad' ELSE 'mismo país, otra ciudad' END AS caso,
       count(*) AS n, round(count(*) / 300000 * 100, 2) AS pct
FROM t JOIN customers c USING (customer_id) GROUP BY 1 ORDER BY n DESC""").to_string(index=False))

 pct_mismo_pais  pct_misma_ciudad  ciudades_tx  ciudades_cliente
          95.45             95.09           28                16


                    caso      n   pct
mismo país, misma ciudad 256695 85.57
 mismo país, ciudad nula  28712  9.57
               otro país  13650  4.55
 mismo país, otra ciudad    943  0.31


In [15]:
hr = q("SELECT hora, count(*) n FROM transactions GROUP BY 1 ORDER BY 1")
dw = q("SELECT dayname(transaction_date) AS dia, isodow(transaction_date) AS d, count(*) n FROM transactions GROUP BY ALL ORDER BY d")
print("hora: coef. de variación del conteo =", round(hr.n.std() / hr.n.mean(), 4), "| min/max =", hr.n.min(), hr.n.max())
dw["pct"] = (dw.n / dw.n.sum() * 100).round(2); print(dw.to_string(index=False))
print(q("""SELECT isodow(transaction_date) AS d, count(*) n FROM transactions
           WHERE transaction_date >= '2024-01-01' AND transaction_date < '2024-01-29' GROUP BY 1 ORDER BY 1""").T.to_string())

hora: coef. de variación del conteo = 0.0022 | min/max = 183570 185224
      dia  d      n   pct
   Monday  1 636774 14.39
  Tuesday  2 714188 16.14
Wednesday  3 718522 16.24
 Thursday  4 714610 16.15
   Friday  5 710921 16.07
 Saturday  6 499942 11.30
   Sunday  7 430051  9.72
       0      1      2      3      4      5      6
d      1      2      3      4      5      6      7
n  16095  19113  18650  16370  16526  12873  11622


In [16]:
# ¿el monto (relativo al rango de su tipo y moneda) depende de hora, día, comercio o canal?
from scipy import stats
mono = q(f"""
WITH t AS (SELECT * FROM transactions WHERE {DISPUTABLE_SQL} USING SAMPLE reservoir(300000 ROWS) REPEATABLE (42))
SELECT hora, isodow(transaction_date) AS dia, coalesce(merchant_name, '∅') AS comercio, channel,
       percent_rank() OVER (PARTITION BY transaction_type, currency ORDER BY amount) AS amount_rank
FROM t""")
for col in ["hora", "dia", "comercio", "channel"]:
    g = mono.groupby(col).amount_rank
    h = stats.kruskal(*[x.values for _, x in g])
    print(f"{col:9s} rango de medias del rank de monto: {g.mean().min():.3f}–{g.mean().max():.3f} | Kruskal p={h.pvalue:.3g}")

hora      rango de medias del rank de monto: 0.495–0.506 | Kruskal p=0.827
dia       rango de medias del rank de monto: 0.498–0.503 | Kruskal p=0.305
comercio  rango de medias del rank de monto: 0.487–0.510 | Kruskal p=0.222
channel   rango de medias del rank de monto: 0.496–0.508 | Kruskal p=0.0788


In [17]:
# V de Cramér entre pares categóricos (muestra 300k)
cat = q(f"""SELECT transaction_type, channel, transaction_status, coalesce(merchant_name,'∅') AS merchant,
            transaction_country, customer_segment, hora::VARCHAR AS hora, isodow(transaction_date)::VARCHAR AS dia
            FROM transactions USING SAMPLE reservoir(300000 ROWS) REPEATABLE (42)""")
def cramers_v(a, b):
    ct = pd.crosstab(a, b); chi2 = stats.chi2_contingency(ct, correction=False)[0]
    return np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
cols = cat.columns
cv = pd.DataFrame([[cramers_v(cat[a], cat[b]) if a != b else np.nan for b in cols] for a in cols], index=cols, columns=cols)
print(cv.round(3).to_string())

                     transaction_type  channel  transaction_status  merchant  transaction_country  customer_segment   hora    dia
transaction_type                  NaN    0.004               0.005     0.432                0.003             0.006  0.008  0.004
channel                         0.004      NaN               0.005     0.010                0.005             0.004  0.009  0.005
transaction_status              0.005    0.005                 NaN     0.010                0.003             0.004  0.008  0.004
merchant                        0.432    0.010               0.010       NaN                0.008             0.010  0.009  0.008
transaction_country             0.003    0.005               0.003     0.008                  NaN             0.006  0.009  0.004
customer_segment                0.006    0.004               0.004     0.010                0.006               NaN  0.009  0.005
hora                            0.008    0.009               0.008     0.009              

**Interpretación:** Hay una única señal geográfica fuerte: el 85,6 % de las transacciones ocurre en la ciudad del cliente y el 4,55 % en otro país (el 9,6 % tiene ciudad nula). Sirve para riesgo, pero el cliente rara vez da la ciudad en su reclamo. La hora es plana (CV 0,002). El día de la semana tiene un patrón leve (sábado 11,3 %, domingo 9,7 %, lunes 14,4 %, frente a ~16 % de martes a viernes), coherente con el desfase horario de ~6 h (H11). El monto no depende de la hora, el día, el comercio ni el canal (Kruskal p ≥ 0,08; medias del rank 0,49–0,51), y todas las V de Cramér son < 0,05, salvo la estructural tipo↔comercio. El ranker no podrá aprender nada de patrones temporales o de comportamiento.

## Resumen de la fase 1
- **Real y útil:** 3 años de transacciones consistentes (sin duplicados ni huérfanas), un cliente por transacción, estados y tipos bien formados, geografía coherente con el cliente.
- **Poco realista:** montos y centavos uniformes (sin precios redondos), 24 comercios limpios y solo en compras, canal independiente del tipo, sin hábitos por cliente ni patrones horarios.
- **Lo que más pesa para el ranker:** la densidad. Mediana de 2 candidatas en 60 días y 28 % de casos con una sola. Con montos uniformes con centavos, el monto exacto es casi un identificador. La fase 2 mide cuánta ambigüedad queda con cada pista.